# Kiểm tra độc lập kết quả xử lý: raw → clean → phiên/nhóm → split

Notebook này **chỉ đọc** dữ liệu và kiểm tra lại toàn bộ đầu ra của notebook 01 và 02 bằng cách tính lại độc lập, không dùng lại hàm của hai notebook đó.

- Đầu vào: `data/raw/events.csv`, 8 tệp Parquet trong `data/processed/`, `split_config.json` và hai báo cáo `reports/events_clean/metrics.json`, `reports/session_split/metrics.json`.
- Đầu ra duy nhất: `reports/data_validation/metrics.json` (36 phép kiểm tra, `status: PASS` khi tất cả đạt).
- Không sửa raw, không ghi đè Parquet hay cấu hình. Cuối notebook kiểm tra lại mã băm để chứng minh điều đó.

Chạy sau notebook 01 → 02, bằng kernel của `.venv`. Chạy không tương tác:
`jupyter nbconvert --to notebook --execute --inplace notebooks/kiem_tra_ket_qua_xu_ly.ipynb`.

Notebook cần khoảng 4–6 GB RAM vì giữ cùng lúc raw dạng chuỗi và các bảng đã xử lý.

## 1. Đường dẫn, hàm hỗ trợ và danh sách tệp cần kiểm tra

`check()` ghi kết quả từng phép kiểm tra. Một phép kiểm tra lỗi không dừng ngay để báo cáo thấy được mọi lỗi; cell cuối dừng với lỗi nếu có phép nào `FAIL`.

In [ ]:
from pathlib import Path
import hashlib
import json
import sys
import pandas as pd
import pyarrow as pa

cwd = Path.cwd().resolve()
ROOT = next(
    (path for path in [cwd, *cwd.parents]
     if (path / "data/raw/events.csv").is_file()
     and (path / "data/processed/events_sessionized.parquet").is_file()), None
)
if ROOT is None:
    raise FileNotFoundError("Cần data/raw/events.csv và đầu ra notebook 01/02; mở notebook từ project hoặc notebooks/.")

RAW = ROOT / "data/raw/events.csv"
PROCESSED = ROOT / "data/processed"
CLEAN = PROCESSED / "events_clean.parquet"
SESSIONIZED = PROCESSED / "events_sessionized.parquet"
SESSIONS = PROCESSED / "sessions.parquet"
GROUPS = PROCESSED / "timestamp_groups.parquet"
SPLIT_NAMES = ["train", "validation", "test", "boundary_drop"]
SPLIT_FILES = {name: PROCESSED / "splits" / f"{name}.parquet" for name in SPLIT_NAMES}
SPLIT_CONFIG = PROCESSED / "split_config.json"
CLEAN_REPORT = ROOT / "reports/events_clean/metrics.json"
SPLIT_REPORT = ROOT / "reports/session_split/metrics.json"
REPORT_PATH = ROOT / "reports/data_validation/metrics.json"

COLUMNS = ["timestamp", "visitorid", "event", "itemid", "transactionid"]
CLEAN_COLUMNS = COLUMNS + ["event_id", "event_time"]
REQUIRED = ["timestamp", "visitorid", "event", "itemid"]
ALLOWED_EVENTS = ["view", "addtocart", "transaction"]
INACTIVITY_MS = 30 * 60 * 1000
INPUT_FILES = [RAW, CLEAN, SESSIONIZED, SESSIONS, GROUPS, *SPLIT_FILES.values(), SPLIT_CONFIG]


def rel(path):
    return path.relative_to(ROOT).as_posix()


def fingerprint(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        while block := stream.read(8 * 1024 * 1024):
            digest.update(block)
    return {"bytes": path.stat().st_size, "sha256": digest.hexdigest()}


def content_sha256(frame, key):
    ordered = frame.sort_values(key, kind="stable").reset_index(drop=True)
    digest = hashlib.sha256(json.dumps([str(c) for c in ordered.columns]).encode("utf-8"))
    digest.update(pd.util.hash_pandas_object(ordered, index=False).to_numpy(dtype="<u8").tobytes())
    return digest.hexdigest()


def write_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    try:
        temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n",
                             encoding="utf-8", newline="\n")
        json.loads(temporary.read_text(encoding="utf-8"))
        temporary.replace(path)
    finally:
        temporary.unlink(missing_ok=True)


def frames_equal(left, right, quiet=False):
    try:
        pd.testing.assert_frame_equal(left, right, check_exact=True)
        return True
    except AssertionError as error:
        if not quiet:
            print("  khác biệt:", str(error).splitlines()[0])
        return False


CHECKS = []


def check(name, condition, detail):
    passed = bool(condition)
    CHECKS.append({"check": name, "status": "PASS" if passed else "FAIL",
                   "detail": detail if passed else "KHÔNG ĐẠT: " + detail})
    print("PASS" if passed else "FAIL", "|", name, "|", detail)
    return passed


# Ví dụ biên cho hàm so sánh/băm dùng trong notebook.
_example = pd.DataFrame({"event_id": [2, 1], "value": ["b", "a"]})
assert content_sha256(_example, ["event_id"]) == content_sha256(_example.iloc[::-1], ["event_id"])
assert frames_equal(_example, _example.copy()) and not frames_equal(_example, _example.assign(value="x"), quiet=True)
del _example

fingerprints_before = {rel(path): fingerprint(path) for path in INPUT_FILES}
print("Phiên bản:", sys.version.split()[0], "| pandas:", pd.__version__, "| pyarrow:", pa.__version__)
print("Project:", ROOT.name, "| số tệp đầu vào:", len(fingerprints_before))

## 2. Đọc báo cáo, cấu hình và dữ liệu

Raw được đọc dạng chuỗi như notebook 01 để kiểm tra lại từ biểu diễn gốc.

In [ ]:
clean_report = json.loads(CLEAN_REPORT.read_text(encoding="utf-8"))
split_report = json.loads(SPLIT_REPORT.read_text(encoding="utf-8"))
split_config = json.loads(SPLIT_CONFIG.read_text(encoding="utf-8"))

raw = pd.read_csv(RAW, dtype="string")
clean = pd.read_parquet(CLEAN, engine="pyarrow")
events = pd.read_parquet(SESSIONIZED, engine="pyarrow")
sessions = pd.read_parquet(SESSIONS, engine="pyarrow")
groups = pd.read_parquet(GROUPS, engine="pyarrow")
split_frames = {name: pd.read_parquet(path, engine="pyarrow") for name, path in SPLIT_FILES.items()}
cut80, cut90 = split_config["cutoff_ms"]
print(f"raw {len(raw):,} | clean {len(clean):,} | events {len(events):,} | sessions {len(sessions):,} | groups {len(groups):,}")
print({name: len(frame) for name, frame in split_frames.items()})

## 3. Phiên bản tệp và dòng dõi giữa hai notebook

- `file_versions`: 10 tệp khớp dung lượng và SHA-256 theo byte ghi trong báo cáo; nếu báo cáo có `content_sha256` thì nội dung bảng cũng phải khớp.
- `notebook_lineage`: đầu ra notebook 01 chính là đầu vào notebook 02 và nguồn của cấu hình chia tập.

In [ ]:
expected_files = {
    rel(RAW): clean_report["source"],
    rel(CLEAN): clean_report["output"],
    rel(SPLIT_CONFIG): split_report["split_config"],
    **split_report["artifacts"],
}
frames_by_path = {rel(CLEAN): clean, rel(SESSIONIZED): events, rel(SESSIONS): sessions, rel(GROUPS): groups,
                  **{rel(path): split_frames[name] for name, path in SPLIT_FILES.items()}}
version_ok = set(expected_files) == set(fingerprints_before)
content_checked = 0
for path, expected in expected_files.items():
    actual = fingerprints_before.get(path, {})
    same = actual.get("bytes") == expected.get("bytes") and actual.get("sha256") == expected.get("sha256")
    if "content_sha256" in expected:
        same &= content_sha256(frames_by_path[path], expected["content_key"]) == expected["content_sha256"]
        content_checked += 1
    if not same:
        print("  lệch:", path)
    version_ok &= same
check("file_versions", version_ok,
      f"{len(expected_files)} tệp khớp dung lượng và SHA-256; {content_checked} tệp Parquet khớp content_sha256.")

lineage_ok = (
    clean_report["source"]["sha256"] == fingerprints_before[rel(RAW)]["sha256"]
    and clean_report["output"]["sha256"] == split_report["source"]["sha256"] == split_config["source"]["sha256"]
    and split_report.get("upstream_report_verified") is True
    and split_report["split_config"]["sha256"] == fingerprints_before[rel(SPLIT_CONFIG)]["sha256"]
)
check("notebook_lineage", lineage_ok, "Đầu ra 01, đầu vào 02 và cấu hình chia tập cùng một phiên bản.")

## 4. Raw → clean

Tính lại bảng sạch từ raw theo đúng quy tắc: bỏ trùng tuyệt đối 5 cột gốc (giữ dòng đầu), `event_id` = chỉ số dòng raw, ép kiểu số nguyên và đổi mili-giây sang UTC. So sánh từng giá trị theo `event_id`.

In [ ]:
duplicate_mask = raw.duplicated(subset=COLUMNS, keep="first")
expected_clean = raw.loc[~duplicate_mask].copy()
expected_clean["event_id"] = expected_clean.index.astype("int64")
for column in ["timestamp", "visitorid", "itemid"]:
    expected_clean[column] = expected_clean[column].astype("int64")
expected_clean["transactionid"] = expected_clean["transactionid"].astype("Int64")
expected_clean = expected_clean.set_index("event_id").sort_index()
actual_clean = clean.set_index("event_id").sort_index()

raw_clean_ok = actual_clean.index.equals(expected_clean.index)
if raw_clean_ok:
    for column in ["timestamp", "visitorid", "itemid"]:
        raw_clean_ok &= bool((actual_clean[column].to_numpy() == expected_clean[column].to_numpy()).all())
    raw_clean_ok &= actual_clean["transactionid"].astype("Int64").equals(expected_clean["transactionid"])
    raw_clean_ok &= bool((actual_clean["event"].astype(str).to_numpy()
                          == expected_clean["event"].astype(str).to_numpy()).all())
    raw_clean_ok &= bool((actual_clean["event_time"]
                          == pd.to_datetime(expected_clean["timestamp"], unit="ms", utc=True)).all())
check("raw_to_clean_exact_values", raw_clean_ok, "Mọi dòng/cột sạch khớp raw sau đúng quy tắc bỏ trùng và UTC.")

duplicates_removed = int(duplicate_mask.sum())
check("event_count_balance",
      len(raw) == len(clean) + duplicates_removed
      and clean_report["raw_rows"] == len(raw)
      and clean_report["clean_rows"] == len(clean)
      and clean_report["exact_duplicate_rows_removed"] == duplicates_removed,
      "Dòng gốc = dòng sạch + dòng trùng loại.")

schema_ok = (
    list(clean.columns) == CLEAN_COLUMNS
    and not clean[REQUIRED + ["event_id", "event_time"]].isna().any().any()
    and all(pd.api.types.is_integer_dtype(clean[c].dtype) for c in ["timestamp", "visitorid", "itemid", "event_id", "transactionid"])
    and all(bool(clean[c].dropna().ge(0).all()) for c in ["timestamp", "visitorid", "itemid", "event_id", "transactionid"])
)
check("clean_schema", schema_ok, "Không thiếu trường bắt buộc.")

is_purchase = clean["event"].astype(str).eq("transaction")
check("purchase_ids",
      clean.loc[is_purchase, "transactionid"].notna().all() and clean.loc[~is_purchase, "transactionid"].isna().all(),
      "Lượt mua đều có mã giao dịch.")
check("valid_events", clean["event"].astype(str).isin(ALLOWED_EVENTS).all(), "Không có nhãn hành vi khác.")
check("clean_uniqueness", clean["event_id"].is_unique and not clean.duplicated(subset=COLUMNS).any(),
      "event_id duy nhất; không còn dòng trùng năm cột.")
check("utc",
      isinstance(clean["event_time"].dtype, pd.DatetimeTZDtype) and str(clean["event_time"].dt.tz) == "UTC"
      and bool((clean["event_time"] == pd.to_datetime(clean["timestamp"], unit="ms", utc=True)).all()),
      "Thời gian giữ UTC, không tự cộng 7 giờ.")
del raw, expected_clean, actual_clean, duplicate_mask, is_purchase

## 5. Clean → phiên

Kiểm tra hai chiều quy tắc phiên trên các sự kiện sắp theo visitor → timestamp → event_id:
cùng visitor và nghỉ ≤ 30 phút thì **phải** cùng phiên; đổi visitor hoặc nghỉ > 30 phút thì **phải** khác phiên.

In [ ]:
check("clean_to_sessionized_exact_values",
      frames_equal(events[CLEAN_COLUMNS].sort_values("event_id").reset_index(drop=True),
                   clean.sort_values("event_id").reset_index(drop=True)),
      "Thêm khóa phiên/nhóm/split không làm đổi hay mất sự kiện sạch.")

ordered = events.sort_values(["visitorid", "timestamp", "event_id"]).reset_index(drop=True)
same_visitor = ordered["visitorid"].eq(ordered["visitorid"].shift())
gap = ordered["timestamp"] - ordered["timestamp"].shift()
same_session_as_previous = ordered["session_id"].eq(ordered["session_id"].shift())
must_continue = same_visitor & gap.le(INACTIVITY_MS)
must_break = (~same_visitor | gap.gt(INACTIVITY_MS)) & ordered.index.to_series().gt(0)
check("session_no_excess_splits", same_session_as_previous[must_continue].all(),
      "Cùng visitor, nghỉ ≤30 phút thì cùng phiên.")
check("session_required_splits", (~same_session_as_previous[must_break]).all() and ordered["session_id"].is_monotonic_increasing,
      "Đổi visitor hoặc nghỉ >30 phút thì khác phiên.")
check("one_visitor_per_session", events.groupby("session_id")["visitorid"].nunique().eq(1).all(),
      "Một phiên chỉ có một visitor.")
del ordered, same_visitor, gap, same_session_as_previous, must_continue, must_break

expected_sessions = events.groupby("session_id").agg(
    visitorid=("visitorid", "first"), start_ms=("timestamp", "min"), end_ms=("timestamp", "max"),
    n_events=("event_id", "size"), n_groups=("group_id", "nunique"),
)
actual_sessions = sessions.set_index("session_id").sort_index()
summary_ok = sessions["session_id"].is_unique and actual_sessions.index.equals(expected_sessions.index)
if summary_ok:
    for column in expected_sessions.columns:
        summary_ok &= bool((actual_sessions[column].to_numpy() == expected_sessions[column].to_numpy()).all())
    summary_ok &= bool((actual_sessions["start_time"] == pd.to_datetime(actual_sessions["start_ms"], unit="ms", utc=True)).all())
    summary_ok &= bool((actual_sessions["end_time"] == pd.to_datetime(actual_sessions["end_ms"], unit="ms", utc=True)).all())
check("session_summary", summary_ok, "Bảng phiên khớp toàn bộ sự kiện và mỗi phiên có một dòng.")

pairs_by_split = sessions.groupby("split", observed=True)["n_prefix_target_pairs"].sum()
query_ok = sessions["n_prefix_target_pairs"].eq(sessions["n_groups"] - 1).all() and all(
    int(pairs_by_split.get(name, 0)) == split_report["splits"][name]["prefix_target_pairs_before_candidate_filter"]
    for name in SPLIT_NAMES)
check("session_query_count", query_ok, "Số cặp dự kiến = số nhóm − 1; chưa tạo mẫu thực tế.")
del expected_sessions, actual_sessions

## 6. Nhóm timestamp

Một `(session_id, timestamp)` đúng một nhóm; nhóm không lẫn phiên/thời gian; `group_index` bắt đầu từ 1 và liên tiếp theo timestamp trong phiên.

In [ ]:
check("one_group_per_session_timestamp",
      events.groupby(["session_id", "timestamp"])["group_id"].nunique().eq(1).all(),
      "Không tách sự kiện đồng thời thành nhiều nhóm.")

expected_groups = events.groupby("group_id").agg(
    session_id=("session_id", "first"), visitorid=("visitorid", "first"), timestamp=("timestamp", "first"),
    n_events=("event_id", "size"), n_items=("itemid", "nunique"), n_behaviors=("event", "nunique"),
)
actual_groups = groups.set_index("group_id").sort_index()
group_summary_ok = groups["group_id"].is_unique and actual_groups.index.equals(expected_groups.index)
if group_summary_ok:
    for column in expected_groups.columns:
        group_summary_ok &= bool((actual_groups[column].to_numpy() == expected_groups[column].to_numpy()).all())
    group_summary_ok &= bool((actual_groups["event_time"] == pd.to_datetime(actual_groups["timestamp"], unit="ms", utc=True)).all())
check("group_summary", group_summary_ok, "Mỗi nhóm có một dòng, số liệu khớp events.")
del expected_groups, actual_groups

check("group_one_session", events.groupby("group_id")["session_id"].nunique().eq(1).all(),
      "Không gom chung các phiên khác nhau.")
check("group_one_timestamp", events.groupby("group_id")["timestamp"].nunique().eq(1).all(),
      "Không gom sự kiện khác timestamp.")

ordered_groups = groups.sort_values(["session_id", "timestamp"]).reset_index(drop=True)
expected_index = ordered_groups.groupby("session_id").cumcount() + 1
check("group_order",
      bool((ordered_groups["group_index"].to_numpy() == expected_index.to_numpy()).all())
      and not ordered_groups.duplicated(["session_id", "timestamp"]).any(),
      "group_index bắt đầu từ 1 và liên tiếp theo timestamp trong phiên.")
del ordered_groups, expected_index

group_lookup = groups.set_index("group_id")
check("event_group_links",
      bool((events["group_index"].to_numpy() == events["group_id"].map(group_lookup["group_index"]).to_numpy()).all())
      and bool((events["session_id"].to_numpy() == events["group_id"].map(group_lookup["session_id"]).to_numpy()).all()),
      "Thứ tự nhóm trong events khớp bảng nhóm.")
del group_lookup

## 7. Mốc chia và nhãn split

Tính lại phân vị 80%/90% (nội suy tuyến tính, cắt về mili-giây nguyên) trên **toàn bộ events sạch**, trước khi loại phiên qua biên. Phiên được gán split theo toàn phiên.

In [ ]:
recomputed_cutoffs = [int(clean["timestamp"].quantile(q, interpolation="linear")) for q in [0.8, 0.9]]
check("time_cutoffs",
      recomputed_cutoffs == split_config["cutoff_ms"] and split_config["quantiles"] == [0.8, 0.9]
      and split_config["inactivity_threshold_ms"] == INACTIVITY_MS,
      "Mốc chia khớp phân vị 80%/90% trước loại phiên qua biên.")

session_split = sessions["split"].astype(str)
crosses = pd.Series(False, index=sessions.index)
for cutoff in [cut80, cut90]:
    crosses |= sessions["start_ms"].le(cutoff) & sessions["end_ms"].gt(cutoff)
check("boundary_drop",
      crosses.eq(session_split.eq("boundary_drop")).all()
      and int(crosses.sum()) == split_report["splits"]["boundary_drop"]["sessions"],
      "Đúng toàn bộ phiên qua một hoặc hai mốc.")

train_s = sessions.loc[session_split.eq("train")]
validation_s = sessions.loc[session_split.eq("validation")]
test_s = sessions.loc[session_split.eq("test")]
check("train_time_range", len(train_s) > 0 and train_s["end_ms"].le(cut80).all(),
      "Toàn phiên trong train đúng khoảng thời gian.")
check("validation_time_range",
      len(validation_s) > 0 and validation_s["start_ms"].gt(cut80).all() and validation_s["end_ms"].le(cut90).all(),
      "Toàn phiên trong validation đúng khoảng thời gian.")
check("test_time_range", len(test_s) > 0 and test_s["start_ms"].gt(cut90).all(),
      "Toàn phiên trong test đúng khoảng thời gian.")

split_by_session = sessions.set_index("session_id")["split"].astype(str)
check("event_split_links",
      bool((events["split"].astype(str).to_numpy() == events["session_id"].map(split_by_session).to_numpy()).all()),
      "Events dùng nhãn split của đúng phiên.")
check("group_split_links",
      bool((groups["split"].astype(str).to_numpy() == groups["session_id"].map(split_by_session).to_numpy()).all()),
      "Groups dùng nhãn split của đúng phiên.")

file_sessions = {name: set(frame["session_id"].unique()) for name, frame in split_frames.items()}
disjoint = all(file_sessions[a].isdisjoint(file_sessions[b])
               for i, a in enumerate(SPLIT_NAMES) for b in SPLIT_NAMES[i + 1:])
check("session_not_shared",
      disjoint and events.groupby("session_id", observed=True)["split"].nunique().eq(1).all(),
      "Không phiên nào ở nhiều tập.")
del file_sessions

## 8. Từng tệp split

Mỗi tệp phải chứa **đúng** các sự kiện của những phiên mang nhãn đó trong bảng phiên — không thiếu, không thừa, mọi cột giống bảng tổng. Tổng 4 tệp bằng events sạch; train trước validation trước test.

In [ ]:
for name in SPLIT_NAMES:
    wanted_sessions = sessions.loc[session_split.eq(name), "session_id"]
    expected_partition = events.loc[events["session_id"].isin(wanted_sessions)]
    check(f"{name}_file_exact_values",
          frames_equal(split_frames[name].sort_values("event_id").reset_index(drop=True),
                       expected_partition.sort_values("event_id").reset_index(drop=True))
          and split_frames[name]["split"].astype(str).eq(name).all(),
          f"Toàn bộ dòng/cột tệp {name} khớp các sự kiện cần có.")
del wanted_sessions, expected_partition

all_ids = pd.concat([frame["event_id"] for frame in split_frames.values()], ignore_index=True)
check("partition_balance",
      len(all_ids) == len(clean) and all_ids.is_unique and set(all_ids) == set(clean["event_id"]),
      "Tổng ba tập hợp lệ và boundary_drop bằng events sạch.")
del all_ids

check("strict_time_order",
      split_frames["train"]["timestamp"].max() < split_frames["validation"]["timestamp"].min()
      and split_frames["validation"]["timestamp"].max() < split_frames["test"]["timestamp"].min()
      and train_s["end_ms"].max() < validation_s["start_ms"].min()
      and validation_s["end_ms"].max() < test_s["start_ms"].min(),
      "Train trước validation trước test.")

## 9. Đầu vào giữ nguyên và ghi báo cáo

Băm lại mọi tệp đầu vào. Báo cáo chỉ có `status: PASS` khi cả 36 phép kiểm tra đạt; nếu có lỗi, báo cáo vẫn được ghi để xem chi tiết rồi notebook dừng với lỗi.

In [ ]:
fingerprints_after = {rel(path): fingerprint(path) for path in INPUT_FILES}
check("inputs_unchanged", fingerprints_after == fingerprints_before,
      "Mọi tệp dữ liệu/cấu hình được kiểm tra giữ nguyên.")

failed = [item["check"] for item in CHECKS if item["status"] != "PASS"]
report = {
    "generated_at_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "status": "PASS" if not failed else "FAIL",
    "python_version": sys.version.split()[0],
    "pandas_version": pd.__version__,
    "pyarrow_version": pa.__version__,
    "notebook": "notebooks/kiem_tra_ket_qua_xu_ly.ipynb",
    "scope": "Raw events -> clean -> sessions/groups -> temporal split files. Metadata, query/target and model metrics are validated in later notebooks.",
    "counts": {
        "raw_rows": clean_report["raw_rows"],
        "duplicates_removed": duplicates_removed,
        "clean_rows": len(clean),
        "sessions": len(sessions),
        "timestamp_groups": len(groups),
        "split_events": {name: len(frame) for name, frame in split_frames.items()},
    },
    "n_checks": len(CHECKS),
    "checks": CHECKS,
    "inputs": fingerprints_before,
}
write_json(REPORT_PATH, report)
print("Báo cáo:", rel(REPORT_PATH), "| status:", report["status"], "| số kiểm tra:", len(CHECKS))
if failed:
    raise AssertionError(f"Có phép kiểm tra không đạt: {failed}")
assert len(CHECKS) == 36